### Imports

In [ ]:
import torch
from tqdm import tqdm
import os
from torcheval.metrics import BinaryAUROC
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append("../utils")
from utils import load_model_and_tokenizer, load_prompts, apply_chat_template
from construct_predictors import collect_hidden_activations, Predictor

torch.set_grad_enabled(False)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

### Sparse LLaMA2

In [ ]:
model_path="../../models/SparseLLM/prosparse-llama-2-7b"
model, tokenizer = load_model_and_tokenizer(model_path, attn_implementation="eager", torch_dtype=torch.float16, device_map="cuda:0")
prompts = load_prompts("test_prompts.json")
prompts = apply_chat_template(prompts, tokenizer)
mlp_inputs, gate_outputs, down_inputs = collect_hidden_activations(model, tokenizer, prompts)
model=model.to("cpu")
intermediate_size, hidden_size=model.model.layers[0].mlp.gate_proj.weight.shape
n_layers = len(model.model.layers)
metric = BinaryAUROC(num_tasks=intermediate_size)

In [ ]:
mlp_inputs[0].shape

In [4]:
class PowerInferPredictor(torch.nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int, rank: int):
        super().__init__()
        self.fc1 = torch.nn.Linear(hidden_size, rank, bias=False, dtype=torch.float16)
        self.fc2 = torch.nn.Linear(rank, intermediate_size, bias=False, dtype=torch.float16)
        self.act_fn = torch.nn.ReLU()
    def forward(self, x):
        return self.fc2(self.act_fn(self.fc1(x)))
    
class SVDPredictor(torch.nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int, rank: int):
        super().__init__()
        self.fc1 = torch.nn.Linear(hidden_size, rank, bias=False, dtype=torch.float16)
        self.fc2 = torch.nn.Linear(rank, intermediate_size, bias=True, dtype=torch.float16)
    def forward(self, x):
        return self.fc2(self.fc1(x))
dejaVuPredictor=torch.nn.Sequential(
    torch.nn.Linear(hidden_size, 1024, bias=False, dtype=torch.float16),
    torch.nn.ReLU(),
    torch.nn.Linear(1024, intermediate_size, bias=False, dtype=torch.float16)
)

class PolarSparsityPredictor(torch.nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int, rank: int):
        super().__init__()
        self.fc1 = torch.nn.Linear(hidden_size, rank, bias=False, dtype=torch.float16)
        self.fc2 = torch.nn.Linear(rank, intermediate_size, bias=False, dtype=torch.float16)
        self.activation = torch.nn.Identity()
    def forward(self, x):
        return self.fc2(self.activation(self.fc1(x)))

In [5]:
powerinfer_path="../../models/SparseLLM/prosparse-llama-2-7b-predictor"
svd_predictors_256_path="../weights/ablation/sparse_llama/data_whitening_r256_s0.5"
svd_predictors_1024_path="../weights/ablation/sparse_llama/data_whitening_r1024_s0.8"
dejavu_path="../predictors/dejavu_training/checkpoint-1024"
polar_sparsity_path="../../Polar-Sparsity/checkpoint/prosparse-llama-2-7b-routers/mlp"
svd_predictors_256_naive_path="../weights/ablation/sparse_llama/naive_svd_r256_s0.5"
svd_predictors_1024_naive_path="../weights/ablation/sparse_llama/naive_svd_r1024_s0.8"

In [ ]:
polar_1024_roc_auc_scores = []
for layer_id in tqdm(range(n_layers)):
    weights = model.model.layers[layer_id].mlp.gate_proj.weight
    input_data = mlp_inputs[layer_id].to(model.device)
    true_values = model.model.layers[layer_id].mlp.act_fn(gate_outputs[layer_id].to(model.device))
    true_sparsity_pattern = (true_values > 0.0).to(torch.int16).T
    # polar sparsity, r=1024
    predictor = PolarSparsityPredictor(hidden_size=hidden_size, intermediate_size=intermediate_size, rank=1024)
    predictor.load_state_dict(torch.load(os.path.join(polar_sparsity_path,f"mlp_router_{layer_id}.pt"), weights_only=True))
    pred_values=predictor(input_data).T
    metric.reset()
    metric.update(pred_values, true_sparsity_pattern)
    polar_1024_roc_auc_scores.append(metric.compute().mean().item())
    print(polar_1024_roc_auc_scores)



In [ ]:
print(polar_1024_roc_auc_scores)

In [ ]:
svd_predictors_256_naive_roc_auc_scores = []
svd_predictors_1024_naive_roc_auc_scores = []
svd_predictors_256_roc_auc_scores = []
svd_predictors_1024_roc_auc_scores = []
powerinfer_1024_roc_auc_scores = []
dejavu_1024_roc_auc_scores = []
for layer_id in tqdm(range(n_layers)):
    weights = model.model.layers[layer_id].mlp.gate_proj.weight
    input_data = mlp_inputs[layer_id].to(model.device)
    true_values = model.model.layers[layer_id].mlp.act_fn(gate_outputs[layer_id].to(model.device))
    true_sparsity_pattern = (true_values > 0.0).to(torch.int16).T

    # naive svd, r=256
    predictor = SVDPredictor(hidden_size=hidden_size, intermediate_size=intermediate_size, rank=256)
    predictor.load_state_dict(torch.load(os.path.join(svd_predictors_256_naive_path,f"model_{layer_id}.pt"), weights_only=True))
    pred_values=predictor(input_data).T
    metric.reset()
    metric.update(pred_values, true_sparsity_pattern)
    svd_predictors_256_naive_roc_auc_scores.append(metric.compute().mean().item())

    # naive svd, r=1024
    predictor = SVDPredictor(hidden_size=hidden_size, intermediate_size=intermediate_size, rank=1024)
    predictor.load_state_dict(torch.load(os.path.join(svd_predictors_1024_naive_path,f"model_{layer_id}.pt"), weights_only=True))
    pred_values=predictor(input_data).T
    metric.reset()
    metric.update(pred_values, true_sparsity_pattern)
    svd_predictors_1024_naive_roc_auc_scores.append(metric.compute().mean().item())

    # data aware svd, r=256
    predictor = SVDPredictor(hidden_size=hidden_size, intermediate_size=intermediate_size, rank=256)
    predictor.load_state_dict(torch.load(os.path.join(svd_predictors_256_path,f"model_{layer_id}.pt"), weights_only=True))
    pred_values=predictor(input_data).T
    metric.reset()
    metric.update(pred_values, true_sparsity_pattern)
    svd_predictors_256_roc_auc_scores.append(metric.compute().mean().item())

    # data aware svd, r=1024
    predictor = SVDPredictor(hidden_size=hidden_size, intermediate_size=intermediate_size, rank=1024)
    predictor.load_state_dict(torch.load(os.path.join(svd_predictors_1024_path,f"model_{layer_id}.pt"), weights_only=True))
    # pred_values=(input_data @ predictor.fc1.weight.T @ predictor.fc2.weight.T).T
    pred_values=predictor(input_data).T
    metric.reset()
    metric.update(pred_values, true_sparsity_pattern)
    svd_predictors_1024_roc_auc_scores.append(metric.compute().mean().item())

    # powerinfer, r=1024
    predictor = PowerInferPredictor(hidden_size=hidden_size, intermediate_size=intermediate_size, rank=1024)
    predictor.load_state_dict(torch.load(os.path.join(powerinfer_path,f"model_{layer_id}.pt"), weights_only=True))
    # pred_values=(torch.nn.functional.relu(input_data @ predictor.fc1.weight.T) @ predictor.fc2.weight.T).T
    pred_values=predictor(input_data).T
    metric.reset()
    metric.update(pred_values, true_sparsity_pattern)
    powerinfer_1024_roc_auc_scores.append(metric.compute().mean().item())

    # dejavu, r=1024
    dejaVuPredictor.load_state_dict(torch.load(os.path.join(dejavu_path,f"layer{layer_id}.pt"), weights_only=True))
    # pred_values=(torch.nn.functional.relu(input_data @ predictor.fc1.weight.T) @ predictor.fc2.weight.T).T
    pred_values=dejaVuPredictor(input_data).T
    metric.reset()
    metric.update(pred_values, true_sparsity_pattern)
    dejavu_1024_roc_auc_scores.append(metric.compute().mean().item())
    print(svd_predictors_256_naive_roc_auc_scores, svd_predictors_1024_naive_roc_auc_scores, svd_predictors_256_roc_auc_scores, svd_predictors_1024_roc_auc_scores, powerinfer_1024_roc_auc_scores, dejavu_1024_roc_auc_scores)


In [ ]:
print(svd_predictors_256_naive_roc_auc_scores, svd_predictors_1024_naive_roc_auc_scores, svd_predictors_256_roc_auc_scores, svd_predictors_1024_roc_auc_scores, powerinfer_1024_roc_auc_scores, dejavu_1024_roc_auc_scores)

In [9]:
svd_predictors_256_naive_roc_auc_scores = [0.9133492294659145, 0.8693647352199015, 0.8349367591196156, 0.8153685565278115, 0.8364839599523249, 0.8483098470049107, 0.8604091471041575, 0.8755693191979967, 0.8790804157017392, 0.8833376338435677, 0.877348749504075, 0.8790602284227268, 0.875666800871633, 0.8810957710244353, 0.8835429685238307, 0.8893712250317303, 0.9017033565563517, 0.9033780781169859, 0.9051098581166765, 0.9090926251396358, 0.9168378600343563, 0.9197943696837075, 0.9240583878806812, 0.9218365234409444, 0.9237882969075011, 0.9255724670130342, 0.9279498603772481, 0.9272791565576924, 0.93023430650314, 0.9339614913751284, 0.9361337979754363, 0.9406850655873235] 
svd_predictors_1024_naive_roc_auc_scores = [0.9752784459117173, 0.9591327395797113, 0.931915284083848, 0.9141839953836477, 0.9235879879141458, 0.929879246534621, 0.9353470817663471, 0.9424345629477244, 0.9439281065643026, 0.9473117126311776, 0.9444245922768115, 0.945652437103338, 0.9446817082606397, 0.9463698180858148, 0.9475287506247558, 0.9501584854741687, 0.957362152710099, 0.9585305874698276, 0.9598838949627577, 0.9615219599250526, 0.9653039580877504, 0.9667552967129511, 0.9683799704191737, 0.9672797348481079, 0.9677467626673608, 0.9680182852013426, 0.9685745064538906, 0.9680392600229589, 0.9689872675276066, 0.9697618035107876, 0.9702530074176287, 0.9724574542117731] 
svd_predictors_256_roc_auc_scores = [0.9482863891239615, 0.9220300503698011, 0.9050903088574938, 0.8908297196608802, 0.9014565478570747, 0.9104867316895455, 0.9159311941598616, 0.924765958429193, 0.9264479845198773, 0.9291798555098143, 0.9257620406538398, 0.9254442091388239, 0.9244858479912081, 0.9276551258249223, 0.9269834089581183, 0.9317607190682353, 0.9393937692016341, 0.942169494374464, 0.9459332507576592, 0.9494829486771595, 0.9535151255767089, 0.9555217528978039, 0.9580798742231724, 0.9569541922966371, 0.9575731202195188, 0.9587957308311815, 0.9598645549742079, 0.9600928365894362, 0.9612421118862609, 0.9626178252703173, 0.9633494513990386, 0.9652586322387341] 
svd_predictors_1024_roc_auc_scores = [0.9845475950869592, 0.9768093469352319, 0.9657413970525265, 0.9553551262082318, 0.9595102977688271, 0.9643240424861538, 0.9662937933449313, 0.9697203798874273, 0.9701650981159652, 0.9719400851548222, 0.9705249078623992, 0.970106347136173, 0.9696849083245591, 0.9706417331891162, 0.9701468207804339, 0.972196275334086, 0.9760865724007487, 0.9772181324757283, 0.9792386335954866, 0.9802138875444371, 0.9820513415630928, 0.9826652224522046, 0.9835985157080773, 0.9827463274274258, 0.9825984796902, 0.9827397785814023, 0.9830507362352029, 0.9829430531368009, 0.9833577705865849, 0.9836769314049959, 0.9841849037633441, 0.9852808979306378] 
powerinfer_1024_roc_auc_scores = [0.9684219604276462, 0.9701834517688931, 0.9505952735629642, 0.9365747730524185, 0.9400760381167858, 0.9446032452655511, 0.9500507719412133, 0.9554658824488427, 0.955582837901534, 0.9531497821962632, 0.953403089460753, 0.9541423231189388, 0.9533068319706054, 0.9526048948399165, 0.9515641835544603, 0.9523306594790472, 0.9540028588617829, 0.9562537558531234, 0.9573755488608953, 0.9539254599699739, 0.9503565048932053, 0.9478508417624857, 0.9514931809721794, 0.9497347424700257, 0.9478215375720944, 0.9510567496218955, 0.9469400909156044, 0.9487806812154095, 0.9491600479934197, 0.9503780155872523, 0.9581512307867984, 0.9749857564777589] 
dejavu_1024_roc_auc_scores = [0.8801898811947364, 0.8745602368214013, 0.8373949142678988, 0.8216574009076972, 0.8277641017897142, 0.85111026801103, 0.8547480672410664, 0.859313504534104, 0.8598997817345589, 0.8571962105246438, 0.8553587742432173, 0.8592401150034343, 0.8554915476925059, 0.8600868660732608, 0.8670658384323656, 0.8738369662121112, 0.8839626900263696, 0.8892184305393105, 0.8847854393425132, 0.8946228508110283, 0.8972173556845892, 0.9051957996387178, 0.908703543603642, 0.9098537410648893, 0.9101678049120102, 0.915063173823578, 0.9137192990287705, 0.9173290507457935, 0.9191552406515755, 0.9145667779585294, 0.9206259826526638, 0.9242695800777255]
polar_1024_roc_auc_scores = [0.8183856661373343, 0.8421737718657707, 0.8839217918535058, 0.8542422926096941, 0.8650086262148879, 0.9087142701578586, 0.8893654130867669, 0.8929621091884883, 0.9147531649481045, 0.9009655776914793, 0.9181131470741225, 0.8928616061373386, 0.914916600727546, 0.9165858014554812, 0.9219908688149686, 0.927424172496598, 0.932705241481187, 0.9356296667204926, 0.9300889718560975, 0.9283585334499587, 0.938185521406456, 0.9484792426965502, 0.9497375751461143, 0.9430464816903429, 0.9448433900427502, 0.9510203005566621, 0.9525078778063373, 0.9531044135893686, 0.9551157686391112, 0.9565150332290593, 0.9550126615043715, 0.9317731784212011]
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(6,5))
ms = 5.0
plt.plot(dejavu_1024_roc_auc_scores, "D-", markersize=ms,label="Deja Vu, rank 1024",color="red")
plt.plot(polar_1024_roc_auc_scores, "*-", markersize=ms,label="Polar Sparsity, rank 1024",color="black")
plt.plot(powerinfer_1024_roc_auc_scores, "s-", markersize=ms, label="PowerInfer, rank 1024",color="orange")
plt.plot(svd_predictors_256_roc_auc_scores, "v-", markersize=ms, label="SVDP, rank 256",color="blue")
plt.plot(svd_predictors_1024_roc_auc_scores, "^-", markersize=ms, label="SVDP, rank 1024",color="green")
plt.grid()
plt.ylabel("ROC AUC Score")
plt.xlabel("Layer ID")
plt.legend()
plt.tight_layout()
plt.savefig("predictors-roc-auc-comparison.pdf", format="pdf")  

In [ ]:
plt.figure(figsize=(6,5))
ms = 5.0

plt.plot(svd_predictors_256_naive_roc_auc_scores, "v:", markersize=ms, label="Naive SVD, rank 256",color="blue")
plt.plot(svd_predictors_256_roc_auc_scores, "v-", markersize=ms, label="Data-aware SVD, rank 256",color="blue")
plt.plot(svd_predictors_1024_naive_roc_auc_scores, "^:", markersize=ms, label="Naive SVD, rank 1024",color="green")
plt.plot(svd_predictors_1024_roc_auc_scores, "^-", markersize=ms, label="Data-aware SVD, rank 1024",color="green")


# Get current legend handles and labels
handles, labels = plt.gca().get_legend_handles_labels()
# Define the desired order (indexes of handles/labels)
order = [2,3, 0, 1]  # Here: Linear, Quadratic, Cubic
# Apply reordered legend
plt.legend([handles[i] for i in order], [labels[i] for i in order])


plt.grid()
plt.ylabel("ROC AUC Score")
plt.xlabel("Layer ID")
plt.tight_layout()
plt.savefig("ablation-whitening-roc-auc.pdf", format="pdf")  

### Distribution Shift

In [ ]:
layer_id=0
weights = model.model.layers[layer_id].mlp.gate_proj.weight
input_data = mlp_inputs[layer_id].to(model.device)
true_values = model.model.layers[layer_id].mlp.act_fn(gate_outputs[layer_id].to(model.device))
true_sparsity_pattern = (true_values > 0.0).to(torch.int16).T

# naive svd, r=256
predictor = SVDPredictor(hidden_size=hidden_size, intermediate_size=intermediate_size, rank=256)
predictor.load_state_dict(torch.load(os.path.join(svd_predictors_256_naive_path,f"layer_{layer_id}.pt"), weights_only=True))
# pred_values=(input_data @ predictor.fc1.weight.T @ predictor.fc2.weight.T).T
pred_values=predictor(input_data).T

In [ ]:
torch.topk(torch.mean(pred_values*true_sparsity_pattern,dim=0), k=10,largest=False)

In [ ]:
from matplotlib import pyplot as plt

neuron_id = 6100
active_distr=pred_values[:,neuron_id][true_sparsity_pattern[:,neuron_id]].detach().cpu().numpy()
inactive_distr=pred_values[:,neuron_id][~true_sparsity_pattern[:,neuron_id]].detach().cpu().numpy()



plt.figure(figsize=(5,4))

plt.hist(active_distr, bins=25, density=True, alpha=0.5,label="Active neuron values")
plt.hist(inactive_distr, bins=25, density=True, alpha=0.5,label="Inactive neuron values")
plt.grid()
plt.xlabel("SVD prediction value")
plt.ylabel("Density")
plt.legend(loc='upper left')
plt.savefig("plot.pdf", format="pdf")  
plt.tight_layout()

### Penalty correlations

In [2]:
def estimate_sparsity(gate_outputs) -> np.array:
    threshold = 1e-8

    sparsities = [
                    (a <= threshold).to(torch.float16).mean(dim=-1)
                    for a in gate_outputs
                ] # (n_layers, n_tokens)
    sparsities_mean = np.array([
                    a.mean().item()
                    for a in sparsities
                ]) * 100
    sparsities_p10 = np.array([
                    np.quantile(a,q=0.1).item()
                    for a in sparsities
                ]) * 100
    sparsities_p90 = np.array([
                    np.quantile(a,q=0.9).item()
                    for a in sparsities
                ]) * 100
    
    return sparsities_mean, sparsities_p10, sparsities_p90

In [ ]:
model_path="../../models/SparseLLM/prosparse-llama-2-7b"
model, tokenizer = load_model_and_tokenizer(model_path, attn_implementation="eager", torch_dtype=torch.float16, device_map="cuda:0")
prompts = load_prompts("../calibration_prompts.json")
prompts = apply_chat_template(prompts, tokenizer)
mlp_inputs, gate_outputs, down_inputs = collect_hidden_activations(model, tokenizer, prompts)
sparsities_mean, sparsities_p10, sparsities_p90 = estimate_sparsity(gate_outputs)
model=model.to("cpu")
intermediate_size, hidden_size=model.model.layers[0].mlp.gate_proj.weight.shape
n_layers = len(model.model.layers)

In [4]:
def monotonicity_kendall_tau(arr):
    values1 = np.array(arr)
    values2 = np.arange(len(values1))  # Ideal: [0,1,2,...,n-1]
    return normalised_kendall_tau_distance(values1, values2)

def normalised_kendall_tau_distance(values1, values2):
    n = len(values1)
    assert len(values2) == n, "Both lists have to be of equal length"
    i, j = np.meshgrid(np.arange(n), np.arange(n))
    a = np.argsort(values1)
    b = np.argsort(values2)
    ndisordered = np.logical_or(
        np.logical_and(a[i] < a[j], b[i] > b[j]), 
        np.logical_and(a[i] > a[j], b[i] < b[j])
    ).sum()
    return 1 - ndisordered / (n * (n - 1))

In [ ]:
argsr=1024
kendal_distances_mean = []
kendal_distances_p10 = []
kendal_distances_p90 = []
for layer_id in tqdm(range(n_layers)):
    mlp=model.model.layers[layer_id].mlp
    mlp_input = mlp_inputs[layer_id]
    gate_output = gate_outputs[layer_id]
    down_input = down_inputs[layer_id]
    n_neurons = gate_output[0].shape[-1]
    hidden_size = mlp_input[0].shape[-1]
    w_gate = mlp.gate_proj.weight
    w_down = mlp.down_proj.weight
    act_fn = mlp.act_fn
    r = argsr
    ablation_config = {
            "penalty" : "full",
        }
    compute_dtype = torch.float64
    compute_device = "cpu"
    n_neurons = w_gate.shape[0]
    n_tokens = mlp_input.shape[0] // 2 # half of the dataset for S construction and other for bias calibration

    ### AB construction
    X = mlp_input[:n_tokens].to(compute_device).to(compute_dtype)
    w = w_gate.to(compute_device).to(compute_dtype)
    S = X.T @ X
    S = torch.linalg.cholesky(S, upper=False)
    u, s, v = torch.linalg.svd(w @ S)
    v = torch.linalg.solve(S.T, v.T).T
    del S

    down_proj = (v * (s**0.0).unsqueeze(1))[:r]
    up_proj = (u[:, :r] * (s[:r] ** 1.0).unsqueeze(0))
    del u,s,v
    torch.cuda.empty_cache()

    ### Bias calibration

    X = mlp_input[n_tokens:].to(compute_device).to(compute_dtype) # (n_tokens, d)
    predicted_values = (X @ (down_proj.T @ up_proj.T)).T # (D, n_tokens)

    match ablation_config["penalty"]:
        case "gate":
            neuron_importance = act_fn(gate_output[n_tokens:]).T.to(compute_device).to(compute_dtype) # (D, n_tokens)
        case "gate/up":
            neuron_importance = down_input[n_tokens:].T.to(compute_device).to(compute_dtype) # (D, n_tokens)
        case "full":
            down_norms = torch.linalg.norm(w_down.to(compute_device).to(compute_dtype), dim=0, keepdim=True).T # (D, 1)
            neuron_importance = down_input[n_tokens:].T.to(compute_device).to(compute_dtype) * down_norms # (D, n_tokens)
        case _:
            raise ValueError("Unknown penalty type value!")


    del X
    torch.cuda.empty_cache()

    sort_indices = torch.argsort(predicted_values, dim=-1) # (D, n_tokens)
    sorted_neuron_importance = torch.gather(neuron_importance, dim=-1, index=sort_indices).to(torch.float32) # (D, n_tokens)
    penalty_full = torch.cumsum(sorted_neuron_importance**2, dim=-1) # (D, n_tokens)

    target_size = 256
    multiplier = penalty_full.shape[-1] // target_size
    indices = torch.linspace(0, penalty_full.shape[-1] - 1, target_size).to(torch.int64)
    penalty = penalty_full[:, indices]  # (D, target_size)

    delta_penalty = penalty[:,1:]-penalty[:,:-1]
    delta_penalty = torch.concat([delta_penalty, torch.ones(n_neurons, 1)*float('inf')], dim=-1)
    res = []
    for neuron_id in tqdm(range(n_neurons)): 
        res.append(normalised_kendall_tau_distance(delta_penalty[neuron_id].detach().cpu().numpy(), indices.detach().cpu().numpy()))
    # res = np.nan_to_num(np.array(res))    
    kendal_distances_mean.append(np.mean(res))
    kendal_distances_p10.append(np.quantile(res,q=0.1).item())
    kendal_distances_p90.append(np.quantile(res,q=0.9).item())
    print(kendal_distances_mean[-1])

In [ ]:
plt.figure(figsize=(5,4))
x = range(n_layers)

ms = 5.0
plt.plot(kendal_distances_mean, "s-",  markersize=ms, label="Kendall distance",color="green")
plt.fill_between(
    x,
    kendal_distances_p10,
    kendal_distances_p90,
    color="green",
    alpha=0.1,
)

plt.plot(sparsities_mean/100, "D-", markersize=ms, label="Sparsity ratio",color="blue")
plt.fill_between(
    x,
    sparsities_p10/100,
    sparsities_p90/100,
    color="blue",
    alpha=0.1,
)




plt.grid(True)
plt.xlabel("Layer ID")
plt.ylabel("Normalized value")
# plt.ylim([0, 1])
plt.xlim([0, 31])
plt.legend()
plt.tight_layout()
plt.savefig("kendall.pdf", format="pdf")  